In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

#### 전처리 대상 데이터 가져오기

##### 민원사무처리기준표 데이터를 기준으로 서류사전 구축

In [ ]:
raw_data = pd.read_excel('../P1_Data/01.External/민원사무처리기준표(2023-2-15).xlsx',header = 0) #엑셀데이터 형태상 header지정
data = raw_data.copy()
data

In [ ]:
use_col = data.loc[:,['분류번호','사무명','소관부처','구비서류']] #사용컬럼 지정
use_col = use_col[use_col['구비서류'].notnull()] # 널값제거
use_col

#### 서류항목 전처리 

##### 줄바꿈을 기준으로 row 늘리기

In [ ]:
def split_rows(df, col_name):
    new_df = df.copy()
    new_df[col_name] = new_df[col_name].apply(lambda x: x.split('\n'))  # 지정한 컬럼을 줄바꿈 기준으로 split하여 리스트로 저장
    new_df = new_df.explode(col_name)# 리스트를 행으로 분리(explode)
    new_df = new_df[new_df[col_name].notnull()]# 값이 없는 행 제거
    return new_df

In [ ]:
split_data = split_rows(use_col,'구비서류') #함수실행

In [ ]:
split_data.drop_duplicates(inplace=True) #중복제거
split_data

In [ ]:
target_str = "< 민원인 제출서류 >" # 타켓 값
is_mandatory = False

data = {'민원인_서류': []} # 필요한 데이터를 저장할 딕셔너리 초기화
for line in split_data['구비서류']:
    if target_str in line:
        is_mandatory = True  # target_str 문자열이 포함된 행을 찾으면 is_mandatory를 True로 설정
    elif is_mandatory and line.startswith("<"): # '<' 문자로 시작하는 행을 찾으면 is_mandatory를 False로 설정하고 반복문 다음으로 이동
        is_mandatory = False
        continue
    elif is_mandatory and line.strip():
        data['민원인_서류'].append(line.strip())# is_mandatory가 True이면서 문자열이 있는 행을 찾으면 해당 문자열을 '민원인_서류' 키의 리스트에 추가

result = pd.DataFrame(data)
result

In [ ]:
result = result.drop(result[result['민원인_서류'] == '1. 유형없음'].index) # 1. 유형없음에 해당되는 값 삭제
result

In [ ]:
target_values = result['민원인_서류'].to_list() # '민원인_서류' 컬럼의 값들을 리스트로 변환하여 target_values 변수에 저장
pre_rowdata = split_data[split_data['구비서류'].isin(target_values)] # '구비서류' 컬럼의 값이 target_values 리스트에 포함된 행만 추출하여 pre_rowdata 변수에 저장
pre_rowdata

In [ ]:
pre_rowdata.to_excel('../P1_Data/02.Pretreatment/민원사무처리기준표_predata.xlsx',index=False)

In [ ]:
pre_rowdata['사무명'].drop_duplicates()